# Gravitino connector Flink example

In [1]:
from pyflink.table import EnvironmentSettings, TableEnvironment
from pyflink.common import Configuration
from pyflink.table.expressions import col
from pyflink.table import DataTypes
import os

os.environ["FLINK_ENV_JAVA_OPTS"] = (
    "--add-opens=java.base/java.net=ALL-UNNAMED "
    "--add-opens=java.base/java.lang=ALL-UNNAMED "
    "--add-opens=java.base/java.util=ALL-UNNAMED "
    "--add-opens=java.base/java.io=ALL-UNNAMED"
)

jar_dir = "/tmp/gravitino/flink/packages/"

jar_files = [
    f"file:///{os.path.join(jar_dir, f).replace(os.sep, '/')}"
    for f in os.listdir(jar_dir)
    if f.endswith(".jar")
]

jar_files = ";".join(jar_files)

config = Configuration()
config.set_string("pipeline.classpaths", jar_files)
config.set_string("table.catalog-store.kind", "gravitino")
config.set_string("table.catalog-store.gravitino.gravitino.uri", "http://gravitino:8090")
config.set_string("table.catalog-store.gravitino.gravitino.metalake", "metalake_demo")

env_settings = EnvironmentSettings.new_instance().with_configuration(config)
table_env = TableEnvironment.create(env_settings.in_batch_mode().build())

print(table_env.list_catalogs())

/opt/conda/envs/flink-py310/lib/python3.10/site-packages/apache_beam/runners/portability/stager.py:63: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


['catalog_fileset', 'catalog_hive', 'catalog_hudi', 'catalog_iceberg', 'catalog_iceberg_s3', 'catalog_kafka', 'catalog_mysql', 'catalog_paimon', 'catalog_paimon_s3', 'catalog_postgres', 'default_catalog']


### Flink conector Hive catalog

In [2]:
table_env.use_catalog("catalog_hive")
table_env.execute_sql("CREATE DATABASE IF NOT EXISTS Reading_System")
table_env.execute_sql("USE Reading_System")
table_env.execute_sql("""
    CREATE TABLE IF NOT EXISTS books (
        id INT,
        title STRING,
        author STRING,
        publish_date STRING
    )
""")

In [3]:
result = table_env.execute_sql("SHOW DATABASES")
with result.collect() as results:
    for row in results:
        print(row)

<Row('default')>
<Row('reading_system')>
<Row('sales')>


## Flink write Paimon table in Minio

In [6]:
table_env.use_catalog("catalog_paimon_s3")
result = table_env.execute_sql("SHOW DATABASES")
with result.collect() as results:
    for row in results:
        print(row)

<Row('default')>
<Row('test')>


In [7]:
table_env.execute_sql("CREATE DATABASE IF NOT EXISTS test")
table_env.execute_sql("USE test")

In [8]:
table_env.execute_sql("""
    CREATE TABLE IF NOT EXISTS paimon_table_a (
        aa BIGINT,
        bb BIGINT
    )WITH(
        'type'='paimon',
        's3.access-key'='minioadmin',
        's3.secret-key'='minioadmin',
        's3.endpoint'='http://minio:9000',
        's3.path.style.access'='true'
    );
""")

In [9]:
table_env.execute_sql("INSERT INTO paimon_table_a(aa,bb) VALUES(1,2)")

In [10]:
result = table_env.execute_sql("SELECT * FROM paimon_table_a")
with result.collect() as results:
    for row in results:
        print(row)

<Row(1, 2)>
